In [0]:
%python
# ==============================================================================
# SCRIPT DE EDA / DIAGNÓSTICO DE ENGENHARIA - CAMADA BRONZE (Databricks)
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    StringType,
)

# ------------------------------------------------------------------------------
# 1. Leitura Direta da Tabela do Catálogo
# ------------------------------------------------------------------------------
NOME_TABELA = "credito_prd.bronze.give_me_some_credit_raw"

df_bronze = spark.table(NOME_TABELA)

# ------------------------------------------------------------------------------
# 2. Diagnóstico Estrutural Básico
# ------------------------------------------------------------------------------
total_linhas = df_bronze.count()
total_colunas = len(df_bronze.columns)

print("=" * 60)
print(f"DIAGNÓSTICO INICIAL - TABELA: {NOME_TABELA}")
print(f"Total de Registros (Linhas): {total_linhas:,}")
print(f"Total de Atributos (Colunas): {total_colunas}")
print("=" * 60)

print("\n--- SCHEMA DETECTADO ---")
df_bronze.printSchema()

# ------------------------------------------------------------------------------
# 3. Análise de Nulos, Brancos e Completude
# ------------------------------------------------------------------------------
print("\n--- AUDITORIA DE COMPLETUDE E NULIDADE ---")

expressoes_nulos = []
for coluna in df_bronze.columns:
    tipo_dado = df_bronze.schema[coluna].dataType

    if isinstance(tipo_dado, StringType):
        condicao_invalida = (
            F.col(f"`{coluna}`").isNull()
            | (F.trim(F.col(f"`{coluna}`")) == "")
            | (F.lower(F.trim(F.col(f"`{coluna}`"))).isin("nan", "null", "none"))
        )
    elif isinstance(tipo_dado, (FloatType, DoubleType)):
        condicao_invalida = F.col(f"`{coluna}`").isNull() | F.isnan(F.col(f"`{coluna}`"))
    else:
        # Tipos Timestamp, Date, Integer, Long, Boolean aceitam apenas isNull()
        condicao_invalida = F.col(f"`{coluna}`").isNull()

    expressoes_nulos.append(
        F.count(F.when(condicao_invalida, 1)).alias(f"{coluna}_nulos")
    )

df_metricas_nulos = df_bronze.select(expressoes_nulos).collect()[0].asDict()

relatorio_nulos = []
for col_nome in df_bronze.columns:
    qtd_nulos = df_metricas_nulos.get(f"{col_nome}_nulos", 0)
    pct_nulos = (qtd_nulos / total_linhas * 100) if total_linhas > 0 else 0.0
    relatorio_nulos.append(
        (col_nome, str(df_bronze.schema[col_nome].dataType), qtd_nulos, round(pct_nulos, 2))
    )

df_diagnostico_nulos = spark.createDataFrame(
    relatorio_nulos, ["coluna", "tipo_dado", "qtd_nulos_ou_vazios", "pct_nulos"]
)

df_diagnostico_nulos.orderBy(F.col("pct_nulos").desc()).show(
    n=total_colunas, truncate=False
)

# ------------------------------------------------------------------------------
# 4. Avaliação de Duplicidade (Total e por Chave Primária)
# ------------------------------------------------------------------------------
print("\n--- AUDITORIA DE DUPLICIDADE ---")

linhas_distintas = df_bronze.distinct().count()
linhas_duplicadas_completas = total_linhas - linhas_distintas
pct_duplicatas = (
    (linhas_duplicadas_completas / total_linhas * 100) if total_linhas > 0 else 0.0
)

print(f"Linhas 100% idênticas duplicadas: {linhas_duplicadas_completas:,} ({pct_duplicatas:.2f}%)")

# Verificação de unicidade na chave identificadora
coluna_pk = "customer_id"
if coluna_pk in df_bronze.columns:
    df_chaves_duplicadas = (
        df_bronze.groupBy(coluna_pk)
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.col("count").desc())
    )
    qtd_chaves_repetidas = df_chaves_duplicadas.count()
    print(f"Contagem de chaves '{coluna_pk}' duplicadas: {qtd_chaves_repetidas:,}")
    if qtd_chaves_repetidas > 0:
        print(f"Exemplo de ocorrências duplicadas em '{coluna_pk}':")
        df_chaves_duplicadas.show(5, truncate=False)

# ------------------------------------------------------------------------------
# 5. Cardinalidade e Valores Únicos
# ------------------------------------------------------------------------------
print("\n--- CARDINALIDADE POR COLUNA ---")

expressoes_cardinalidade = [
    F.approx_count_distinct(F.col(f"`{c}`")).alias(c) for c in df_bronze.columns
]
resumo_cardinalidade = (
    df_bronze.select(expressoes_cardinalidade).collect()[0].asDict()
)

dados_cardinalidade = [
    (col, val, round((val / total_linhas * 100), 2) if total_linhas > 0 else 0.0)
    for col, val in resumo_cardinalidade.items()
]

df_cardinalidade = spark.createDataFrame(
    dados_cardinalidade, ["coluna", "valores_distintos_aprox", "pct_distintos"]
)
df_cardinalidade.orderBy(F.col("valores_distintos_aprox").asc()).show(
    n=total_colunas, truncate=False
)

# ------------------------------------------------------------------------------
# 6. Sumário Estatístico e Extremos (Colunas Numéricas de Negócio)
# ------------------------------------------------------------------------------
print("\n--- SUMÁRIO DE DISTRIBUIÇÃO NUMÉRICA ---")

colunas_numericas = [
    f"`{c.name}`"
    for c in df_bronze.schema.fields
    if isinstance(c.dataType, (IntegerType, LongType, FloatType, DoubleType))
    and c.name not in ["customer_id"]  # Omitir chave de identificação do sumário estatístico
]

if colunas_numericas:
    df_bronze.select(colunas_numericas).summary(
        "count", "min", "25%", "50%", "75%", "max"
    ).show(truncate=False)

# ------------------------------------------------------------------------------
# 7. Amostragem Visual dos Dados Brutos
# ------------------------------------------------------------------------------
print("\n--- AMOSTRA DOS DADOS BRUTOS (TOP 5) ---")
display(df_bronze.limit(5))

grafico

In [0]:
%python
# ==============================================================================
# EXIBIR TABELA DE DIAGNÓSTICO DE NULOS PARA VISUALIZAÇÃO GRÁFICA
# ==============================================================================
display(df_diagnostico_nulos)

Databricks visualization. Run in Databricks to view.